# 一、Python注解（Annotations）
注解是为函数参数、返回值或变量添加的元数据（描述数据的数据），不影响代码执行，仅用于类型提示、文档生成或第三方工具解析。
1. 语法
    - 参数注解：参数名: 注解内容
    - 返回值注解：-> 注解内容

In [1]:
# 为函数参数和返回值添加注解
def greet(name: str, age: int) -> str:
    return f"我是{name}，今年{age}岁"

# 查看注解（通过 __annotations__ 属性）
print(greet.__annotations__)
# 输出: {'name': <class 'str'>, 'age': <class 'int'>, 'return': <class 'str'>}

# 调用函数（注解不影响执行）
print(greet("张三", 25))  # 输出: 我是张三，今年25岁

{'name': <class 'str'>, 'age': <class 'int'>, 'return': <class 'str'>}
我是张三，今年25岁


# 二、Python装饰器（Decorators）
装饰器是一种设计模式，用于在不修改原函数代码的前提下，动态扩展函数的功能（如日志记录、性能计时、权限校验等）。
1. 工作原理
    - 装饰器本质是一个高阶函数（接受函数作为参数，返回一个新函数）。通过 @装饰器名 语法糖，可将装饰器应用到目标函数上。

- 案例：性能计时装饰器

In [2]:
import time

# 定义装饰器：记录函数执行时间
def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()  # 开始时间
        result = func(*args, **kwargs)  # 执行原函数
        end = time.time()  # 结束时间
        print(f"{func.__name__} 执行耗时: {end - start:.2f}秒")
        return result
    return wrapper

# 应用装饰器（@timer 等价于 slow_func = timer(slow_func)）
@timer
def slow_func(seconds):
    time.sleep(seconds)  # 模拟耗时操作
    print("函数执行完成")

# 调用函数
slow_func(2)
# 输出:
# 函数执行完成
# slow_func 执行耗时: 2.00秒

函数执行完成
slow_func 执行耗时: 2.01秒


# 三、闭包装饰器
闭包装饰器是基于闭包实现的装饰器。
1. 什么是闭包？
    - 闭包是指内部函数引用了外部函数的变量，且外部函数返回了内部函数。此时内部函数会 “记住” 外部函数的变量状态，即使外部函数已执行完毕。
2. 闭包装饰器的实现
    - 利用闭包的 “状态保持” 特性，可实现更强大的装饰器（如记录函数调用次数、缓存结果等）。

- 案例：记录函数调用次数的闭包装饰器

In [3]:
def count_calls(func):
    count = 0  # 闭包变量：记录调用次数

    def wrapper(*args, **kwargs):
        nonlocal count  # 声明使用外部函数的变量
        count += 1
        print(f"{func.__name__} 已调用 {count} 次")
        return func(*args, **kwargs)

    return wrapper

# 应用装饰器
@count_calls
def add(a, b):
    return a + b

# 调用函数
print(add(1, 2))  # 输出: add 已调用 1 次 \n 3
print(add(3, 4))  # 输出: add 已调用 2 次 \n 7
print(add(5, 6))  # 输出: add 已调用 3 次 \n 11

add 已调用 1 次
3
add 已调用 2 次
7
add 已调用 3 次
11


# （一）装饰器的本质
1. 在python中，函数就是对象，可以：
    - 赋值给变量
    - 作为参数传递给其他函数
    - 作为其他函数的返回值

In [4]:
def greet(name):
    return f"你好, {name}!"

# 1. 函数赋值给变量
say_hello = greet
print(say_hello("张三"))  # 输出: 你好, 张三!

# 2. 函数作为参数
def call_func(func, arg):
    return func(arg)

print(call_func(greet, "李四"))  # 输出: 你好, 李四!

# 3. 函数作为返回值
def get_greeter():
    def inner_greet(name):
        return f"Hello, {name}!"
    return inner_greet

greeter = get_greeter()
print(greeter("王五"))  # 输出: Hello, 王五!

你好, 张三!
你好, 李四!
Hello, 王五!


2. 高阶函数：接受 函数作为参数 或 返回函数 的函数，就是高阶函数。

3. 闭包：闭包是指内部函数记住了外部函数的变量和作用域，即使外部函数已经执行完毕。

In [5]:
def outer(x):
    def inner(y):
        return x + y  # inner引用了outer的变量x
    return inner

closure = outer(10)
print(closure(5))  # 输出: 15（记住了x=10）

15


# （二）装饰器的完整工作原理
装饰器本质上就是一个高阶函数 + 闭包的组合体，它接受一个函数作为输入，返回一个新的函数，新函数通常会在原函数执行前后添加额外功能。
1. 装饰器手动实现（不使用@语法糖）

In [6]:
# 原函数
def add(a, b):
    return a + b

# 定义装饰器函数
def my_decorator(func):
    def wrapper(*args, **kwargs):
        # 1. 原函数执行前的操作
        print(f"准备执行函数: {func.__name__}")
        print(f"参数: args={args}, kwargs={kwargs}")

        # 2. 执行原函数
        result = func(*args, **kwargs)

        # 3. 原函数执行后的操作
        print(f"函数执行完毕，结果: {result}")

        return result
    return wrapper

# "装饰"原函数
add = my_decorator(add)

# 调用被装饰后的函数
print(add(2, 3))

准备执行函数: add
参数: args=(2, 3), kwargs={}
函数执行完毕，结果: 5
5


2. 使用 @语法糖：Python 提供了 @装饰器名 语法，让装饰器的应用更简洁。

In [7]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"执行前: {func.__name__}")
        result = func(*args, **kwargs)
        print(f"执行后: {func.__name__}")
        return result
    return wrapper

# 使用@语法糖，等价于: add = my_decorator(add)
@my_decorator
def add(a, b):
    return a + b

@my_decorator
def multiply(a, b):
    return a * b

print(add(2, 3))
print(multiply(2, 3))

执行前: add
执行后: add
5
执行前: multiply
执行后: multiply
6


# （三）装饰器关键细节：*args 和 **kwargs
注意在 wrapper 函数中，我们使用了 *args 和 **kwargs：这样装饰器就可以通用地装饰任何函数，无论该函数有多少参数。
- *args：接收任意数量的位置参数
- **kwargs：接收任意数量的关键字参数

# （四）函数元数据丢失问题
使用装饰器有一个副作用：被装饰后的函数，它的元数据（如函数名、文档字符串）会变成 wrapper 函数的元数据。

In [8]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        """这是wrapper的文档"""
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def add(a, b):
    """这是add函数的文档，用于计算两数之和"""
    return a + b

print(add.__name__)      # 输出: wrapper  (而不是 add!)
print(add.__doc__)       # 输出: 这是wrapper的文档  (而不是 add的文档!)

wrapper
这是wrapper的文档


- 解决方案：使用functools.wraps
- functools.wraps 会将原函数的元数据复制到包装函数上
- 生产级装饰器的标准写法！

In [9]:
import functools

def my_decorator(func):
    @functools.wraps(func)  # 关键：保留原函数的元数据
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def add(a, b):
    """这是add函数的文档"""
    return a + b

print(add.__name__)  # 输出: add  (正确了!)
print(add.__doc__)   # 输出: 这是add函数的文档  (正确了!)

add
这是add函数的文档


# （五）带参数的装饰器
有时我们需要给装饰器本身传递参数，这需要再包一层函数：

In [10]:
import functools

def repeat(times):  # 第1层：接收装饰器参数
    def decorator(func):  # 第2层：接收被装饰的函数
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(times):
                print(f"第 {i+1} 次执行:")
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

# 使用带参数的装饰器
@repeat(times=3)
def greet(name):
    print(f"你好, {name}!")

greet("张三")

第 1 次执行:
你好, 张三!
第 2 次执行:
你好, 张三!
第 3 次执行:
你好, 张三!


# （六）类装饰器
除了用函数作为装饰器，我们还可以用类作为装饰器。类装饰器通过实现 \_\_call__ 方法来工作。

In [ ]:
import functools

class CountCalls:
    def __init__(self, func):
        self.func = func
        self.count = 0
        # 同样需要保留元数据
        functools.update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"{self.func.__name__} 已被调用 {self.count} 次")
        return self.func(*args, **kwargs)

@CountCalls
def say_hello():
    print("Hello!")

say_hello()
say_hello()
say_hello()

带参数的类装饰器

In [11]:
import functools

class Delay:
    def __init__(self, seconds):
        self.seconds = seconds

    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            import time
            print(f"等待 {self.seconds} 秒...")
            time.sleep(self.seconds)
            return func(*args, **kwargs)
        return wrapper

@Delay(seconds=2)
def greet(name):
    print(f"你好, {name}!")

greet("张三")

等待 2 秒...
你好, 张三!


# （七）多个装饰器的叠加使用
可以将多个装饰器应用到同一个函数上，执行顺序是从下往上（或从内往外）

In [12]:
import functools

def decorator1(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print("装饰器1：执行前")
        result = func(*args, **kwargs)
        print("装饰器1：执行后")
        return result
    return wrapper

def decorator2(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print("装饰器2：执行前")
        result = func(*args, **kwargs)
        print("装饰器2：执行后")
        return result
    return wrapper

# 等价于: my_func = decorator1(decorator2(my_func))
@decorator1
@decorator2
def my_func():
    print("原函数执行")

my_func()

装饰器1：执行前
装饰器2：执行前
原函数执行
装饰器2：执行后
装饰器1：执行后
